# EDA com Full Join das Abas

Este notebook carrega o Excel `BASE DE DADOS PEDE 2024 - DATATHON.xlsx`, faz um *full join* entre as abas `PEDE2022`, `PEDE2023`, `PEDE2024` pela chave `RA` e gera um EDA básico do resultado.

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

file_path = "BASE DE DADOS PEDE 2024 - DATATHON.xlsx"

sheets = ["PEDE2022", "PEDE2023", "PEDE2024"]

# Carrega as abas
frames = {name: pd.read_excel(file_path, sheet_name=name) for name in sheets}

# Garante que a chave exista
missing_key = [name for name, df in frames.items() if "RA" not in df.columns]
if missing_key:
    raise KeyError(f"A chave 'RA' nao existe nas abas: {missing_key}")

# Full join sequencial com sufixos por ano
merged = frames["PEDE2022"].merge(
    frames["PEDE2023"], on="RA", how="outer", suffixes=("_2022", "_2023")
)
merged = merged.merge(
    frames["PEDE2024"], on="RA", how="outer", suffixes=("", "_2024")
)

# EDA basico
print("Dimensoes por aba:")
for name, df in frames.items():
    print(f"- {name}: {df.shape}")

print("\nDimensoes apos full join:", merged.shape)

print("\nAmostra:"
)
display(merged.head())

print("\nInfo:")
merged.info()

# Valores ausentes
missing = merged.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(merged)).round(4)
missing_df = pd.DataFrame({"missing": missing, "missing_pct": missing_pct})

print("\nTop 20 colunas com mais missing:")
display(missing_df.head(20))

# Duplicados de RA
dup_ra = merged["RA"].duplicated().sum()
print(f"\nDuplicados de RA: {dup_ra}")

# Estatisticas descritivas
print("\nDescribe (inclui categorias):")
display(merged.describe(include="all", datetime_is_numeric=True))

# Grafico de missing por coluna (top 20)
plt.figure(figsize=(10, 6))
sns.barplot(
    x=missing_df.head(20).missing.values,
    y=missing_df.head(20).index,
    color="#4c72b0"
)
plt.title("Top 20 colunas com mais valores ausentes")
plt.xlabel("Quantidade de missing")
plt.ylabel("Coluna")
plt.tight_layout()
plt.show()

# Distribuicao de colunas numericas (top 6)
num_cols = merged.select_dtypes(include=["number"]).columns.tolist()
if num_cols:
    sample_cols = num_cols[:6]
    merged[sample_cols].hist(bins=30, figsize=(12, 8))
    plt.suptitle("Distribuicao das principais colunas numericas", y=1.02)
    plt.tight_layout()
    plt.show()

# Frequencia das colunas categoricas (top 5 valores, ate 6 colunas)
cat_cols = merged.select_dtypes(include=["object", "category"]).columns.tolist()
for col in cat_cols[:6]:
    print(f"\nTop 5 valores em {col}:")
    display(merged[col].value_counts(dropna=False).head(5))

Dimensoes por aba:
- PEDE2022: (860, 42)
- PEDE2023: (1014, 48)
- PEDE2024: (1156, 50)

Dimensoes apos full join: (1661, 138)

Amostra:


,RA,Fase_2022,Turma_2022,Nome,Ano nasc,Idade 22,Gênero_2022,Ano ingresso_2022,Instituição de ensino_2022,Pedra 20_2022,...,IPV,IAN,Fase Ideal_2024,Defasagem_2024,Destaque IEG,Destaque IDA,Destaque IPV,Escola,Ativo/ Inativo,Ativo/ Inativo.1
0,RA-1,7.0,A,Aluno-1,2003.0,19.0,Menina,2016.0,Escola Pública,Ametista,...,NaN,10.0,Fase 8 (Universitários),0.0,NaN,NaN,NaN,Universidade Santo Amaro (UNISA),Cursando,Cursando
1,RA-10,7.0,A,Aluno-10,2004.0,18.0,Menina,2021.0,Escola Pública,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,RA-100,4.0,A,Aluno-100,2009.0,13.0,Menina,2019.0,Rede Decisão,Ametista,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,RA-1000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,7.835,10.0,Fase 1 (3° e 4° ano),0.0,NaN,NaN,NaN,EE Helio Luiz Dobrochinski Prof,Cursando,Cursando
4,RA-1001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,7.920,5.0,Fase 2 (5° e 6° ano),-1.0,NaN,NaN,NaN,EE Helio Luiz Dobrochinski Prof,Cursando,Cursando



Info:
<class 'pandas.DataFrame'>
RangeIndex: 1661 entries, 0 to 1660
Columns: 138 entries, RA to Ativo/ Inativo.1
dtypes: datetime64[us](1), float64(77), object(5), str(55)
memory usage: 1.7+ MB

Top 20 colunas com mais missing:


,missing,missing_pct
Destaque IPV.1,1661,1.0
Pedra 23,1661,1.0
Rec Av4_2023,1661,1.0
INDE 23,1661,1.0
Rec Av3_2023,1661,1.0
Rec Av2_2023,1661,1.0
Rec Av1_2023,1661,1.0
Ct_2023,1661,1.0
Cf_2023,1661,1.0
Cg_2023,1661,1.0



Duplicados de RA: 0

Describe (inclui categorias):


TypeError: NDFrame.describe() got an unexpected keyword argument 'datetime_is_numeric'